# Emberline clean rebuild — Stage 2: spatial selection and first-file verification

Stage 1 completed in run **20261114T163518Z_5b1b1c54**. This notebook resumes that run. It reads the saved source inventory, filters daily L3 candidate tiles against the existing Sri Lanka study outline, and downloads and verifies **one** science file.

The regional outline is a frozen project reference to preserve the existing study footprint. Its independent boundary-quality review remains part of the later spatial audit. No previous fire totals, fitted models or validation scores are imported. Production data and the frontend remain unchanged.

The verified Sri Lanka boundary is embedded in section 4, which works in VS Code and browser Colab. Use a CPU runtime and run cells in order.


## 1. Install the same versions recorded by your Stage 1 run


In [ ]:
%pip -q install earthaccess==0.19.0 requests==2.32.4 pandas==2.2.3 numpy==2.1.3 pyarrow==23.0.1 duckdb==1.3.2 shapely==2.2.0 h3==4.5.0 pyhdf==0.11.7 pyproj==3.8.0 scipy==1.16.3


## 2. Mount Drive and resume the existing run

This does not create a new run or rerun Stage 1 queries.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, hashlib, re, urllib.parse
from datetime import datetime, timezone
import importlib.metadata as metadata
import pandas as pd
from shapely.geometry import shape, Polygon

drive.mount('/content/drive')
RUN_ID = '20261114T163518Z_5b1b1c54'
RUN = Path('/content/drive/MyDrive/emberline_clean_rebuild/runs') / RUN_ID
assert RUN.is_dir(), 'The Stage 1 run is not present in this Google Drive account.'
CONFIG = json.loads((RUN/'config.json').read_text())
COLLECTIONS = json.loads((RUN/'inventory/collections.json').read_text())
QUERY_LOG = json.loads((RUN/'inventory/query_log.json').read_text())
expected_versions = json.loads((RUN/'logs/environment.json').read_text())['packages']
actual_versions = {name:metadata.version(name) for name in expected_versions}
assert actual_versions == expected_versions, 'Versions differ. Restart the runtime after installation and rerun.'
STAGE = RUN/'stage2'
STAGE.mkdir(exist_ok=True)
LOCAL = Path('/content/emberline_stage2')/RUN_ID
LOCAL.mkdir(parents=True,exist_ok=True)
print('Resumed:',RUN_ID)
print('Pilot months:',CONFIG['pilot_months'])


## 3. Load the exact saved metadata responses

Every response is checked against the SHA256 recorded in Stage 1. If multiple different snapshots exist for a page, this stage stops for review instead of silently choosing one. Catalogue hits are discovery records, not observing-coverage measurements.


In [ ]:
def saved_entries(product, month):
    pages = {}
    for query in QUERY_LOG:
        url = urllib.parse.urlsplit(query['url'])
        params = urllib.parse.parse_qs(url.query)
        if not url.path.endswith('/granules.json'):continue
        if params.get('collection_concept_id') != [COLLECTIONS[product]]:continue
        if not params.get('temporal',[''])[0].startswith(month+'-01T'):continue
        if 'snapshot' not in query:
            raise ValueError('This query lacks an immutable snapshot reference; review the Stage 1 notebook version.')
        page = int(params.get('page_num',['1'])[0])
        pages.setdefault(page,{})[query['sha256']] = query
    assert pages and sorted(pages)==list(range(1,max(pages)+1)), f'Missing metadata pages: {product} {month}'
    entries=[]
    for page,variants in sorted(pages.items()):
        assert len(variants)==1, f'Multiple metadata snapshots: {product} {month} page {page}. Review before proceeding.'
        query=next(iter(variants.values()))
        filename=query['snapshot']
        assert Path(filename).name==filename
        raw=(RUN/'inventory'/filename).read_bytes()
        assert hashlib.sha256(raw).hexdigest()==query['sha256'], 'Source metadata hash mismatch.'
        entries.extend(json.loads(raw)['feed']['entry'])
    assert len({e['id'] for e in entries})==len(entries), 'Duplicate granule IDs in inventory.'
    return entries

L3 = {(month,product):saved_entries(product,month)
      for month in CONFIG['pilot_months'] for product in ['MOD14A1','MYD14A1']}
print('Verified saved L3 metadata snapshots.')


## 4. Load and verify the Sri Lanka reference outline

Run the next cell directly. It reconstructs the supplied boundary from embedded compressed data and checks its original SHA256. No browser upload widget or local file path is needed. This is a frozen geometry reference; it does not contain fire totals.


In [ ]:
# Self-contained boundary reference: works in VS Code and browser Colab.
import base64, zlib, hashlib, json
from shapely.geometry import shape

BOUNDARY_NAME='Sri_Lanka_pilot_boundary.geojson'
BOUNDARY_SHA256='838db6bf67cb4d21df585d32483c64c9d56556d4fc216564bf4aa982c659382e'
BOUNDARY_COMPRESSED_BASE64 = """
eNqFfcvOBrlx3asIWhsDFlksktkGyCoBAmQZeCHYiiFElgxFXhiG3j3nkM1ikb9ljRaa+Zsfm5e6nLr2v//6z//2L7/99X/51a//
229/8+d//dNv/+sff//73/7Dn3/3xz/8+u9+9ev/s/74/zDgf//7O5QD/uVPf/yX3/7pz7+bQ/7917/7R/xfwt//8Jt/nkP/159+
96v//ps//N/f/Pov+Os//faP//zbP//p3+bYPdv/+Nff//l3//OPv/+3f1rv/Ic//vFP//i7P/zmz+u1+KeNX1pTk/p3v+q/ZBvV
1P7+7361/l6yio35T5vP20it+fNeNY+6ntv6fetWR5r/qA8rxcY1jeVRhj//8fq/55O5tC6io8o3IceU0npu+Xvp+OboSXOvZQ3A
7P17nfU9QHK3XtZfxxpXUhNbU5fsE5n1rmtYn+OyahryLb/4fKpS1gtzEsPJrJ/4C1MfOaU4kTTs0x+nkUTC2RVN1vQ7u+Lj/uYB
nMNqvWFNNbyy4v24pfVraX7go+c+B+gopfoyfEDPaj2FG9Oeaqv7L93HWZJqYXGKE7W0X7gPq+H4pEdCUZuXs8bl6vMNHEqN81kx
cXobnLqsx5NgFAQodS9fzh2WvHc9r0i1j16+M/NdjlGzlXBcitlb+S6/hdf23mQNADHkvgbU4QNyyuuY6igtt2/7hwFa7XuVa5xZ
S/vg8uG3UXHy8Zh6a1n9ePAfzda9SscU38hzG1VTPD2+s0v5cbu1lZHnAN5+/04057Pev0FHgeYgEUBAfifjF8ENlr7peF+JZcz4
HcEcllqu1dluZBOw1LeQ+VxyhiT51m5hnGl3NsY48Cao+GFPkFt2dm9rvkTJ8Bz5kALhM6732tC+r643Utz8OyjD9KVsyJlmXeLv
tfDEX7FiueDG4rq1peLiC4fdi86/43JchtQislZvOOOxz8dXZxBhpfiNc1Yt2fbVNR+HK92vBclLfjkGEgEHdB2XZYi9fUwgZq2T
9Ej3Wfu5j5p6JG2Illpc7l3jsgaWHK2Civ3+BcsqLc7Tqjbn1T4g1GsLrAF6Hr6MPsCZOQfSx/gs4+NEvwUcWK81CJCBZcmWf83P
dSSrl/wbtdax6VPPrQqOSdaxKJn15aSh0KDNOYnnj41oezkf+jBri+QBKnABYh26QWqcpnXN7aUHalkdEukBtCnWHqViHdycF7li
VxA730TFJ4LAK5vYF1eT6vU+zwZS11rWc0iCstm1HIn2n0uHI0hAtKDmoyPGLxUCcmwx5KRqDYKhrOeldvM/D9xsDVuHLMzZztFk
HFlkdPAWFG5/dWMuTfWaB/Ic4vAb18/WTbIEjtEBVdS2MpQzHycIV6fN8OotiFyHJK1dcnivQmk4XIBkB/mvbSfCgU0Mvj+wYd2q
c+1PTbV92qQ2H/e3jjmCL/yvxz1CflLrvGwg5J0oBC1PDfyCNAClRU2EDTz/CydCAEKaXlsYQBf7MvyoerGeZa19AGT53VFBjS1k
+3pecenfnYhPoJCqvf31hagM9Rtaw6oV/cH+FQNru8Yp+OBMQywZjw9iRQ+EhBKGEow/Fyq3/qhOJaC1wItQWAqqCqrFFhNifigk
h44AHdQx4arxwrGlsJMypBik4nzewKvhvan0eArVIJM3WtCDtsAK2eK4Rtn9A4EDudVy3w5+0fbh6wHOwCzFVQXGQfoDCH5/8XFg
3yz3fFCxZs9lQoGPrY/Gd+cFMO1Mo+m+Jag3iL/vFz4M8CX1sCrsueuWOI5R/zbPBOwEcgcwXoMgO7Z+LweoVyVWO1sESoJ1kh7g
B11h7TqJrqSlV7aBhvs+Ucj3/kj9Rmxl1+tAUOI04Hj2r687GiO5SbuYozdA562LAmoFTMiBSs1gagw5KgSqNccDrVXLJhtHWI20
W+s1LuGo+gP6yWD1IoeKU9EfcrtBMsh1jw0KcRPK0XB/c5tRnirpN4xtQh7QH5QNY0kiBxiQoOvRwHkNCq5EyTAmUj826t9437e2
nn7JsI/XnfdSITjm9fDvtTRrJZxXh0YDpolChONA9FrnInB5OPeNvds3AOZ2HvlbJKzETVR7Ath5CsW7CAC6MH1n3+s3gIK9Lc2Y
KUctQvH5vCRLkQDAJ1CW/rhC1rRwWAqZUx09f8OgqmEUxmEQWdryfgxyb2ZBTfMw+oZutnfbIJnqQligGhkOwc6AQhMu3DGOHZJn
RHnCZQPumEZkDuUFPToi785xkHK2nkMpmkTpy/fhclK7Fg5NlPTSjRwHfQpui+sSQNi0kck+TohNDI2QFBD2Q4Z8rBo2Mx8DHOL9
zzkBI0ApRdElsKfqfl3x+VoHPIjLgopooKPvORREkXGUFe2QUY4g3vOAIACr1nMI+bEdPf6c/oyql0WF5Wy6yZtM6NuBAI32LI7T
0nkOcJi33YibCRJ7bQf0Ebe9lOg+z+HjDIzSol0Ghqwuuc2XDe2q3Y+b9gjgxZ7vHJPwzjUI28R1jv3Lfbsgglyz3eYz6HBbMj5M
C+zJZYZDtWwz3HxdQl9a9KYNTl3kPg7AEpit6qfJYVCVLjsGgBSsqGUWAtJsZ1s76wV562dXfgCNf4ZEAJJbf4Z5uzWZnA2At/Jn
TxbQ51k4LCyJviHMC61ZH6mFmxW1z5TsVIb+ADvdwHKtoAIN1k3YZ4IK0XlZrrjHEm5MuLD4nDhkn3T1eUCNLegMjMMkwHC+Uxxa
a98DkNA4R9epb8NOM70e+giiUYCUs8aF3PMUyLyNmvY8/dwQDHTsPj4GHGzJ168QztsKWTcNKi/bwjrDwHY1uvZwv3VsKe5ibIBL
co3eVxwb9JI8Yhi3Drm0BsDCsu0j7D4APwJmiKQJ+y1viezaaRAMDosL65jatkAgU4DVrnsWGP31okgBH2V6S697rMAYWyOJj2vg
vH2fNS/bng9wFH1SNGXUcmnwzzAnR62BHgFTeJuXnsA4mMTRjsS4Splx+TQ5rqcssgbgJ9DIEcZzAPin7gEZJnuNKGfOUPvyZnao
bXdmlP2Y571dOvMg8BJgkxY9fRgHZVirWNhYG1yuXRIb42jOLK/nBDT9cnrPATAsU+Q0jGtlW9B+kngXAVpwDkG30s/yzbcvAooM
1z+f02W/59kHjWVLT5FgaF2V2/vMcRlQ5/KMAbBmpxccbG4XodPF1bcb+GNPjjOo63k+Bv6WLW337sFogH0WHGxQPMBs7aFPjrO1
XFx/Kf532PtyucywiDI2uYLUtNYYAajNcjNfnhII9EB2IHpArRJt//kamMe5xte06YbxeSBjJcfnmNbhffFTVQUBXOMUdrafRqZb
NRIf5MkHIuZeE6TT+hW9NP3HYUsvcu0G2Da5A8HnKXXpjmrASztu5q/JtPJ7EImQ0zDW/efgblskXxdNRH/LtwxpERXQowTD5tzl
SFeICnajuNrzw8Dxi3vi1+swr8+Cm3UJ+V1thV64rM85DvOsx2PLRXAJxGu1a42N/mm5NDzH4dSHXYcB+bIRbTnjoDHzFQSxpbqf
PWExek0H2S/bxMpnOuys2TWdwc7UV3gAokwAfAgGbKZb4dUwrru/eR35vDuL/vK5Plj6OQe1A6MXl7cvS858EOPXfDSW3cFcwh1m
ly1rHE7JAd3ZLxC5rYnoJBlHWuG0ovYHQ2DdZ72QKLbuD6/J2/7oJQz4Yo44FBz2eHlGYGWslRVino3l21+Vl9QZuTs3wIBNGgEX
DiunQ2c4Jo04iw5Tvxb8a/+ok9JjO7lK9gEg+W0W1W+b5DgX9oPBy/UA6s6dlv7+BjJf+AvAxDZwozppEDQlygrIuLHNX/MVgOby
2jdNuVJeCsRfVGIkpsoUyO9CcNBAPHEcNbFekbE5Duq5SAy20qrclsyhGFhBMko4dzpY6jbEzxGDj8p38C2f1wAmLABOb0tp/SfD
pI8eoed7YEwgwRYvhDBWD7nRnI6nCvLN2urLF5numytUbNyiP2c0T/WaB6Tkis7HAQv3XGKIF/IptfMY3PzFmFUOxx79yODies4A
0dFnJLL6Lay0Lc01H0VmkCnfLyWsCHZX1Qg4Fz7tl19mTgB2G1fMHYe7HbpH8QPC5KFXEBvWjBudrlnpvdV20QNEZpVH9wGibGqg
c/37M5QuGPDi8jSAijZf72G0Wb8QNxZRy40dC0yJ0jezEfruBwz4twimQNR0QD6nythbuw6lNLop9+ohGqEtopYpjFVsLeNSD5xc
V2oLjuQ4jlyFMxL25ZAA62iNZjHfUk9wYg0TRkHLA2YYzqgasT00B7B1fRCp4s8H+M1xgPtt+wOdeaAZIABbIHqa8UeWApZAfsUr
hnSja9GfQ5+19UNcbhvhh5JS5BJItTTsAT4gaZxGNL6hO7CbElzyHAbyrpfTQ6SIU66LCoUKt8VDDKIX94j6+5hWskAtnuN82+U2
4ZsAvC5wS7lazvOWmfY0H+D9/udBhLzGCyGhXe4RIW3DVsPKGu3DBprbDwTqJkSg5nOYIu1RS6SOOiSO6wy8yaNcK080HX8T56Ml
Le98UOmpHLrmfL24rF/pJQfqNXpFijlvn3Ewd0KiTqPd3rqDNKhICDWLryH42jfnhFgz42HHDcVxDBLWR/dVSBx3sa73zeNuDztW
YgRZCxfmc5RHjANQgl+POuA4bS4D8czG93d6yPpZJ5j7UGKb+Q5V/TkINUuO0+apW799qB8/cIbmeE3Yb7niPXMYtlrXdYPXyuN9
FmLZCSI2epzrGY6O3AQHWeP29gDYa5tC/E2dhpDGC8hHSuIwMhD6OqYeBBLeb017IMsOxOHBcfHjNoZQj9OnMd/m5DWI0wtUStJI
vp0mt8c2N8PRjzAkxfmok/d6DdfPdLpDvoTYvlmovd43MlpvgTJTT4TytzDwZ2tbEKfuixd/Dy/b4kTQ2+OotX38RpGfSiAb2FHa
3au+h/VGJB64pTEgvNWAKxMoKCxyzQON6OlGbg82AoQa2QSKGRdVHlDU6HJqI1wfZHirbt7sa8HhwqwtgTqAYoCy7NbGECBMnQrU
DyS+Xe54bLW74TUfM+NS04jRrjkOjHyNy2DW7N5s9XHghy7h9HOlJ+fxL3XiszKJRJhD6M6Jsy4o5+2ZnacFsEOP8gOUO+7WUnwf
NEu2oQ+S5T4LqM4gUysQjT1U0yGupBzmx7hGD8B7iyB9aB6ZA4CNxeSdiLHy4Iq0XxjNrjs64uQHTZFnGBcDKt0/+miDXhho6M4v
xpSG4ml8bqx2Ztq0o60wrkAvbFTmHgBym2qL4yCF1X1pftMT9A8OaJ2v1McW7JPGJOwQuHi4OvP1g+NOvsIcBuka6BczaohtGj11
UKe+L/JEXj9MTMzMj1qlfyqvO4X2tmqvcQRUxCmcQ4z5N1q3E9NFCuATDyzcvekkv9trQY/iKBouxBKdp+O5OMiHPtqJvtkvgP7Q
yf6cur/NG4DNBOq/Mlo5AKh5/Q4MfVw9eDm09DwQqGV6rPYDE0j8Gl6o1GPOmy7KYJTYes7EBleQBq0ibU1cIP88TltcNoNwTMNB
KqBVci/BEeHJgpWPYULgWB4FSl1Z+bjMHGC7jxmGurauYRbYA2B9ienPn5rzLIpJhmU6NctjOtFlNM0VDmBGvf2AHRDder2wEmC0
h4GgLKqWGl+oTHtzvMmQbguHBCw4DqxnQKKP+JZMv1C5bSulQR5SWjEMhqyjQpd7CoPXgufZfsGF02x6tAlojaHROK4yyd4eooWK
s3FJG5qmfcsV1wPAbdlzGOb6GFsY2yXnUqlwMWPEcRgoO6Tg4KIUUuWkBsC4auX1OUP8zBy+Q9x0guc7N3lOBKWm67lmp+3CPOf1
M0gj2ftxKQQJA+Eyb5POUFf/brTSUSl1rAGVqdnPi7E2KIQ1Q4ex6xLieCfqwr1GR9VKIAxhzBkdAGNH3QO2ES9f8DfhOihJAgFC
doDkPRgDQFeXYBElAE7PpQDHldoi0UDlMS72vgik1eedAGbDyDheLib5RA5g/FqON4rsVy0QEV1ijjGPWylZzpckFSAhPw9aVRYq
X/gahvfzG+QarHc4iGqOg/Lrj4XOJNwiayLm24/ze8La9XcKgHaFH/lDZcA/7hcgtrmN6/PQeuxxvcK0wX7LvcSTSJG1ZVap1DcI
CDL0+MX31gIx5LG5OjFpUJwyk/H8WBIzStd5JHohPQrNOOYlCxJjzRKrBWaUl5EnC+tMPFT33nlwFqxrwcvHcaCv57bXuD5vOTUa
5Se9gtuyIBFTZWQjPblLkFXmMbi5LVwLttgu0cSsF1BqSHfguKqeg+bZBR386jUCUzQkKIbihRjmyTidWiLOB/7x9GSPdoO5tYSg
rRGCtn1fadbV9DTpAlqpDE/2gajP9+9wjcVxjnjyENAyfl6ZVgJxZiepKPVQnMTnifk8F9LEOBKdWhwHS+skmPt8IFmVcY2DSPGc
KdCABr9TJUJO4kEMT9KCxOxjPWctx/l72T/rPKAnN4gHlpgcVJlOVN1RZef3zOg4x1WZHUCT6/KzMYesJ+sjrtNoq7xJYuCDOlPe
5kT5mKbZB7RyFOCaCNrIf9+hF8yu9TALb4OifX3Uby3kKXMcAI5PA/iudmKqeAzFO1x47NwLop1YDshx0Nl1ywlPRcusqjrmL8fB
FtjFAZ7cZMJESXPhWcmnTCN8UluYKgiUHU8BOtXz/ioTNzTHaUCAzrss2xhpLRekJbtyrm3irEyUCtbLHActsu3a4eMYjl0DqGW2
GPN1gkXAEBqvvTCddlM3NeMIkRKeC9bm61DoI2px/h0421PtIB7NbM3Hjd72XmINH0y574eJsPjhBwXGrzebMlzhVY57gYDksDry
RwE2vGZz+IDSNdShzB3I4SBlXu5I13Or4tVQzedhzKXclNQ9ji5y3jePKIxjXYGnye0TUmLHj5EAsPt48+PolIUQWwNS8SRnf46n
ZRxcMznE3IXpeXGQ+KYhN50LolQZD8cV0txaDx2NujeejWbPCelPgVqbS4ZNCwzetBBOomBm9k715wWG+NIcqRIrPbyamdOsn8Sn
EClPxhwmrz1HhJOYt+75YhkkoyH8R5UCu9yd4Nnn6dMGDSoEtGg7EezkEedBmBogBVVZ38TiJAYJCwRVL5WnjDo+JMSMIkvjmg8L
qb4uoWMhQhjKEBX/fRoMHqwfQkbs3wnvy1I8F+ZZ7ac4R7Dg/LNNzLwfZOZMrwuZLCXPjTLpETy1sNKU9BdETPxZzyFBjMhYmOJx
meYcxzjcAl2w6OpGn3nfbOq4jutmWZI7PKPCX8iKOlsocNAgK5d5ywGF5v+yXHgpt/sDA6pA1l+GEe/ANfQexlQabdEAZCpBk/d9
pTX7DDFWa20+6T4gAwWOZZEyt9fO3zUlCWCeAcO8+ccVHauP1n7KDIfIk4vJralFWtaZjpeiWmax4mA53udd6F5WNCiHTcI+NUNe
9bekG6qn0u4+aFLrrJ96SroYL6+esjrHsRp1/7J7rWtn+uzydkADtfaWojOBpeRoWTOWrrk/9UyAMrW0GiinCqS9FzQNpiNdFwmG
Kr2+9c54EwDf3BmjgurIuZ9y0+GpO3NnjEK2LaQs1remdUSzIt1VwSlxgr1ccvRQwiA79bOa8PLoV+EVSSpvRRlLVqXFcVBz1bfq
68mZvqGwblZ9lnIetz7phrEO7V6ngVeeMuz5nE4pF9q+m8zoln4TYJHtqY+nw70nudZJFs1vgVsZKUt0kWCd1SNppy6YdCzX+QBL
nWocL2pSckGPrmllruKI8bo5jvinBrtwMDfzFPgNzhRkMtBfO/HsfKahnyEuvzKl0X68zhjdj8uqTK/6QUY6497r/mkWnn2B5Np3
QEz1srtYiM4Uv++6ft8gKeWQXyqXe3fQN2/20jszvKpcG5+OYadzWGcjSg6mE48hTz1Yn7H6NQFModouE2btFOZQDhEl+mt1a4N6
CAm3bjmElFgQeZKFTveJARxgMebCun3Tp8YuM+VvhFBJpr9cflTLQ+V71YStcZmtDAIDtfl35k+au1Z9/zjs0lpYDiVCE33Krhlx
ShZDMsLODLW+DAct0iXGW6FzYYL0p89Fn2cq8bhykB7MV0gx2kkF3/d5nJYfAAhNZpyMXpF2KjgrjLq+9p2seeHFOReQrN3rnAWp
V97pHFeNFe1hnVioF2eWMx9Tqy3ESen9M/eHySktbSvQKQA7ubQfXQ1qbq5GnNz05zgqwbVBQAY/kAbSDKUHfAp1EnovpFmvyL83
upDf7VqttZdIDjPs1t8eKpUG4YhcAQWletlJ8yJ7yv0irzG+pIxVd6wO7JpP4/ZvEHN06wV6YL3a2NR3OsRk3JdEqgJz9ezZy4dr
YNC0GGYFtOkpCOfqTqwvLsqQ2unmwWhcTLao9NrXt2VGYx1wLmHZzF/PNaK52dKgsYg4vA6UgWM/3VKI+uffM0NEXjDPpMqx5qWU
3Nhcz/uZIl3DNttMxihPhe10Og2JyQ1kHxcL/kJIgJxTSPFNdAzvHXrrCVglAVSt5B7QsPeP8WFCM2tlETGMNZ5zwRF/SUagfy2h
70Bpq7vR7DdwXptH2g/YQMa7vCTsp+SQ2ThzWa6o+pyYSZtXMpcQrNa3kFUKM4fD9limU2t7e/SQJq78Wowjng0tKnpfD5h4q+XZ
P820mfOVIWhPqYsXDQMvtFXvwZ+LRxFOFbMkvTsdsRDxdFxolBWxhKAYz+UUABtQQzw3Eln7D5oMGe5vrrQAADX3rp82O2xOFXuV
UOeEplc4z5gRyli3bBBzyLmxeG9tA6aK5qd1BotfxrVaSBB3Cp8OItAHduXHF1ZAjav+aM4H27bF9F+eXar1rd0G642VIwi0fzpT
tLMu86YE67YBZW3n5bpKarypq7Zg+jb6eY/le394nj0l67wOkrHdr8vHw+OtI9g5y3OPxxrHFMXyIPTGxPISayZmLLP8EHaV5Soj
3GKmB0rfavHCiHcPNImjyaO/wJfZbkyZCUyGzabydgdoBRbWyjPHLpk1eSYQWcxRWNUt5/rLyu7NMLW9Y1boEMCr+ZqqibqxeciH
AY21IGFW3ilvLx7wW89Zl9nLywsAYTo+cWDMhXvL6aEjaiwbyrCRiheoHnKvI9ceD4glOB6/94NkonSNqZ+ZLltPBjrSBEJy9eQS
1rS0IGWGxjJC2JnY50t2dEvWFsorGMZzf9F5C+8ipvlKpTatT/8FvDKP9Vzp2Hj6iTDVTwL+4DAglS0izyFBca+KOdZKDLkc63Me
pmvFJkgMBiYPZJz1ZFjTMcU/8RL3sgI1ElTNdXOfYwOieriz1mhed/qEyKBv8xr2OEqxOiSxyMFVhATVPTSHgwClmZ7nBaIu9yAN
WGTdi73NdxiusLHyWBmu+nFSld7THgAu86GLHKjB8M9KYIVacwPagS/Y1KuMv0RXY77Da2gANbcSod5o06Z9LVWo8Cwt5lHSrhz9
OfAOQvfAyxpH7OpcfbqMQPelmGHIqmA5b4OKWMsxVhvZQyA901lqV5orO2oEfwjh1JX2CWm2VXbwm0AclBrzbmmnj/MeCocW83zb
6S8jYGGRKyeZhVr62qCFnpIr+ZptfrrcTMjS0CIXDYqeGH+wMUHTH+3h9acfY2YZzKJxwXB9nAiNBdBedr3GsZ9P/dEQBBTeY2NH
kENtp1cgQ2bj4znxupvT5G9QzmrkEZif9sM72FnAprEqD+PUs4xDQxrG3mJZq5CAvcXFcQnARLJ2J933kx10rJvKdPk4rkzseLo4
MYl5CUvau7H7k/eRWFLOqCvyg9sZBopZsRxHRrG3SduP+QCiYbFaaMA5PsCJN/W7I8mySGRD8yzMvX7ENoleogFx62zmRmZNlw7F
f7u/8dyUUm+uAeDbHUI9boiZX9QiCAII8ATqILZotcXabFaQHi+F0Nt5rZf+jfZ0C2KCw4K/III65AcfNfPc/IVvGYuU/Nr0Bcz9
1bbgFr3U8RxPASVrbCxQCjHqy1idWypRKTA+13axnTNGp9CXC3hXBkN+ONuYCqGrIopBFwvePKZyrX6gZL7jZa6arsqmyiYDr5PL
WOS5mpzSS1pemdwYUvoqAOmAfYUue1x8xcDpOMeOU4Z9cK8WJpjHvFj06AiG6SSiAED6E3Z1qpr4IX+1oyxwtMerhXfhiq5aasg3
0XJjpTHbY8ViZqCQ5BE+7+81lEgrUiewAqHXo7zZEtGt/7oK2kU9i8O7qkLzlbsKiT3+tF3p+HMc7mDOA/TEUnv/+2iLDZhzqef9
rCDJsXoJZkzNp3MotnCVfnWmoFV/nlkcF63DzkTW8dh30Bbq1XqrCwIWfzDfma7BFll9HFI+fUkpZVcfmwTW81C9nJ/J0Dg9GFnb
z36pJe++KMKijCs1ZO62zWzrME9ma623ddggHearIyYTCLze9qwrfU6SMev902PH4Vi+KlhC5rw9hc7kY5ZoWuzboS2fnMw9zBgG
6bErh7HfydubbtAPfJkdg7nR+iLvmbYk1/bAcNk7AHikEZIw1RzaEDFII8Vjr7APsvSvsU6tHo9PrADqfXdWrdv08lQf6D+TfrX4
xL0Mn1h7zS22h6QZmnJ+MuJmVDV/HULpZ3nyAnC0LNqIrwG+Fi+2Kz4PDlO/hp1sKPHjueV2Nf2jE/CYwCGqDQjaYwPQOrpXs4iH
x2fb7dh6jq0LPAWzi4+rJlfbLKJbTebPmV4x+01RO3uCB5ECeGH9oBXPOPQUOuo3NQmdkfJ0eNenyxhL05NobGylnFme3C6K7WSx
hxuECOS5PfcOomvt6rJIx4aXIXkak9BgybH/YGZmoKce0iXaemxIVRi4qU5AjV0/Utjf89zobbv6nmEZp6LBQ/644ny1LVO6De3Z
PtNrc19tzegHlmc7ifWqJXb9K+yqI/JkDrJjpBfFrG2xctAzQffpsP1TimSt0MHNu8H5Ldtpo2ZsimxPdgVIdboV2H6VbjZ9BABZ
9Pjy5rjOsnSnT0i0XOI1sV2XSrimketqxknvpJ6DJV3GNoeVKRt27kdMvx6h6oXG4bypL76mqaC87uQ183ZXz0d2piw/6JnVNRrf
y6qq7R72pD4GgFr/unsyYy6WTHDfmZAu3BP7JXeTu7veLAfWsdrTFjupHUKzOWTyzva9Q24rZabnzC4xcbkMObvoPuOwi6sTci0M
a7cnUSuz0UbKd7vfUzvpuSkA7sxZWBOxt6ZPQDHVvwbA7GyQHopi6Mk7Eaz2haPl7BePf2O/jNXZl6lU7T4y6MhTxrB/z2LOmzFp
1VpZlDuLte3JrQXOaGV8jYVxAP1t4Aj+69a+DsXOqKCEstotgyHkJO2d5cG89J+N5OmVJ0GMTWVq0ABMRxxnXYydfgub+POcLNjq
2zElVbuik3MAAebacmV5qE/JTkcatbQyJ9ybiI4zDvT0TcAYvKf5pZmMux5Utl5+m1UmRmmiNFW+L/ebK5hOPK6mzFS9p5O3Zwuy
jrIG5oHByJydvz5NbefjEn4XZbalikpZZ1f9+jBHKSC6UJ86d1ny02yJ42Y/3TgOxKye8Dolx7pRJqXvv7Ihw8XKSteqp4PRc5VD
uJw/pw7153SXlBEJhq1+3W3op8JufB9pdoiAXp+kT7bNK1+faLb+kvykl7Ivo6fTLN4GSjsNpz2tNbF0OXZxp/NYu91KiAqoStwY
qyqbu2f92GBCfG39K73brsNL50kt6chQ7A9yqvR5x66MMOyKuPtdD3WygfvdD7h6TFGP8GNDwsiaxrJUq6/Sa3o6Dn49c8lQL4YV
tvmw1eMVyrx7KpkPmNGeq5cvi1n3RK4MjOmCsW+tMa/QzZSjTZh9kOO6WEDgURfX2Zr7p1IhJfPlsZupoJw6tkaFcU9h96yKHLTd
ieuSOmHwXZ/LVbHnR1zVDH04v7nG7N30aigtQGietE0HUE1RiDGHoLm75iDxLtKuRsUUJr4sH9fZ1jq24jd2c/CEWOBQUG/8Ygbz
Qbo7rQ+w7dmyXD2pC7On9/PC2rf46QBcHSSc7wsCarR873scoguWDCNE13ogy3wasEqtNynRsMgv0GRRXr/6M7ObR9E3a5be6NBr
Zc0XvCn8XkVfE+C02/lzPwmXq1VzT12eD0+w2WzJX/dj6V5I5t7XwZ6f42q93ZWh5scXw1BwvTpDz1L3/IQFYfAVWc2YU5sd/x8z
2xhCiRoHaBeAX06KalbL4RJxY2wDeVJTS2896D+ihJPZM1gYrfGrB202b5TH18suBmycFtdhWrz61M64XFLsW9uYh+Y1Nsfno6Y9
tu/FddSc5c3g1UymiczAZpm1PlF1pk3mpNd569HV7bhZmBb+3e+shfYHSd1LUr6DHl4Lc3xxTBBcOL0xCzz4h9jgQ9cRV/z0CfBC
tDANcD1n3y17s4PZrTLHrwCA+odnvfkOiGgWfUN5FG8gfRxAymKf2EucNTDHgzJbtay/0z7RHydUa/vmB4M0b7nkJAVZ+OFAFm15
4rP7R8e0HeJGyArtRxULfoH/zROb3novh3JWz3ZSietqac4Icb6qrdP8lo57PtY4YQafveoxMW1t9U5PvXtdjUtMNmKuEb4xxK3u
s4FGB8V8C4HES/pi7B+92U9P+cGYx+dPK2PDKjnezPmlmPtLNmD+4CQuzjDLT8jslh25PbfDOveav761vWj5QWb84E6/2gMDoJ0P
/Py1dcatsKvm5zxknPtNDmDJ4NdP6dlFyi2Ge+kOlFmX87jEEzsLxBACO2zZaK9T9q+vJCx3zJjR1QFd9Lg6TlkA1OXUULnX4zwJ
vkd+vC5y1sR9jqCdHNkk95vHmreBOI+N3WMuRxJEpLwOHJo2y0HWaCPZj9VKlhIVGt2lp5/h4UaV1C9/GT2f3iCuHK7tliMozlPW
ueBjjumeZ66r8JsT6ehdHWlcfjRW33lW11n237iM86kDfiUgLX4t3kmXHpjGRN6AKAqkgvdL66fqBdxr8aCZtX9edeYz8RLhNU4h
1O4msP/BasJCIVdKj8CUwMoDev4idgq9zAIlmbkPwxGg3L1HaZgxebWG8pd8fQtN2VFznJ//56sJH7qwfLKyl3qlO9BrIz2iRqFw
4Xx2IhzvV7SgwcU7UC8tzPLwbu/3hSq/xbCha/I0HTvpzP/5ssIO5kegLpQ7Qxfp/agNs/CrXQBhhE/wDJlib6ZrsmTtLdQYbJue
L7zCNvThs2e6VJmR30rsv7+UMkyNtTz2JtvuthPegd0s39HSqNp42o5cZ3glAjWzdKSGBx3pyx7L1wL420zeICG21iUa/ZBGzdM5
w2f5Wr3xaydQzE/OByzCnO8bbwQT5Yn1MvlE6ve6GPSq+ftr6adtgJ4glLS6DMZGWK8nqkaAdZ0GrqDUN4RZGHWwgAzMpoL/EaVj
OcFlYLN5sDcjPuEs9pa5DV7cu7yfZRtETuX6FFRhhYZG99QcB4vMLquksEKjvrfP1Ei9rA5mKHguN1vY55u6Mz8r1B/sSJlh6frm
EXu5tPc4+qw6i6/jf0p9uII6KLXPqGPg5jxQfoBlf9wnQDpGslO9Z67VJR/bR5QWvzFEHep5hkc58gNa1wd1mPV0VBVbl4TbZBLa
+BlXhzLI15dm6L208SSUz87S2q+vh/Fzef46oxftNrTxdi1vdR/D4qbXV5uquSP6LJ8fVumXrKHDSt8iM9ySaI04m3n0xZ6kRibP
eNeAj3YgGKo+IopBmPV1MSOMOJiNtexy+T/4Pd7yfgOLaQwyItDgRxpOp+cR7JzcvnMC6A9gbvp+IxOChD2Y6giVDrO6tYt6Avj5
RCaThL7nROk/ixWVMCBeO7srhvwOduqP3lQj4ji1W8CZ7fsAls22VXdiWacnsV5EbLSfz2fcWJ/9MQdR1ZOlyjrz0uQSWlL8668S
qu9gqK95emavulfn6UzNiOxO5mtyqgqZbrKsdBEJ1URMJQkEyK5SKj+qmP6m7g16mqnry3HAzxrYeD+RBfIZF8SAEjEvpgimRQM4
WtYn/QXyfCt4+m9qCUTEaoTW3oJB2hdFri9uSTudlU4BLOj3chXjZ/20yDiFgzyHcrsPiLvfpDzW4KT4eVnmLXf9kWlG/8j6iBYz
qL1p1kFIQOulfV/xwtm9RRqsx+gtcj2xeCk/vtH4V+8F//zlL3//l/8P+OX6lQ==
"""
boundary_bytes=zlib.decompress(base64.b64decode(BOUNDARY_COMPRESSED_BASE64))
assert hashlib.sha256(boundary_bytes).hexdigest()==BOUNDARY_SHA256, 'Embedded boundary checksum mismatch.'
boundary_json=json.loads(boundary_bytes)
assert boundary_json['type']=='FeatureCollection' and len(boundary_json['features'])==1
REGION=shape(boundary_json['features'][0]['geometry'])
assert REGION.is_valid and REGION.geom_type in ['Polygon','MultiPolygon']
boundary_path=STAGE/BOUNDARY_NAME
if boundary_path.exists():
    assert boundary_path.read_bytes()==boundary_bytes, 'An existing run boundary differs; review before replacing it.'
else:
    boundary_path.write_bytes(boundary_bytes)
print('Reference boundary verified. Bounds:',REGION.bounds)


## 5. Filter the daily-product candidate tiles

CMR bbox searches deliberately return a larger candidate set. This cell uses actual tile footprint polygons. Missing or invalid footprint metadata stops the selection for review; it is not silently discarded. Temporal packages crossing month/year boundaries remain present. Identical daily overlaps will be verified and deduplicated in the processing stage.


In [ ]:
def tile_intersects_region(entry):
    polygons=entry.get('polygons')
    assert polygons, f'Missing footprint metadata for {entry["id"]}'
    intersects=False
    for group in polygons:
        for text in group:
            coordinates=list(map(float,text.split()))
            assert len(coordinates)>=8 and len(coordinates)%2==0
            # CMR polygon coordinates are latitude longitude pairs.
            geometry=Polygon([(coordinates[i+1],coordinates[i]) for i in range(0,len(coordinates),2)])
            assert geometry.is_valid, f'Invalid footprint for {entry["id"]}'
            intersects |= geometry.intersects(REGION)
    return bool(intersects)

selected={};selection_audit=[];selection_summary=[]
for (month,product),entries in L3.items():
    selected_here=[]
    for entry in entries:
        keep=tile_intersects_region(entry)
        selection_audit.append({'month':month,'product':product,'id':entry['id'],'intersects_region':keep})
        if not keep:continue
        links=[link['href'] for link in entry.get('links',[])
            if not link.get('inherited') and link.get('rel','').endswith('/data#')
            and urllib.parse.urlsplit(link.get('href','')).scheme=='https'
            and urllib.parse.urlsplit(link['href']).path.endswith('.hdf')]
        assert len(links)==1, f'Ambiguous science link for {entry["id"]}'
        assert urllib.parse.urlsplit(links[0]).hostname=='data.lpdaac.earthdatacloud.nasa.gov'
        record={'id':entry['id'],'product':product,'name':entry['producer_granule_id'],
            'time_start':entry['time_start'],'time_end':entry['time_end'],
            'catalogue_size_bytes':round(float(entry['granule_size'])*1048576),
            'download_url':links[0]}
        if entry['id'] in selected:assert selected[entry['id']]==record
        selected[entry['id']]=record
        selected_here.append(record)
    selection_summary.append({'month':month,'role':CONFIG['pilot_months'][month],'product':product,
        'bbox_candidates':len(entries),'selected_files':len(selected_here),
        'tiles':','.join(sorted({r['name'].split('.')[2] for r in selected_here}))})

SELECTED=list(selected.values())
manifest={'run_id':RUN_ID,'boundary_sha256':BOUNDARY_SHA256,'source_query_log_sha256':hashlib.sha256((RUN/'inventory/query_log.json').read_bytes()).hexdigest(),
          'files':SELECTED,'selection_audit':selection_audit,'summary':selection_summary,
          'pixel_coverage_verified':False,'production_changes':False}
manifest_bytes=json.dumps(manifest,indent=2).encode()
manifest_path=STAGE/'selected_science_files.json'
if manifest_path.exists():assert manifest_path.read_bytes()==manifest_bytes, 'Selection changed; review the run before proceeding.'
else:manifest_path.write_bytes(manifest_bytes)
display(pd.DataFrame(selection_summary))
print('Unique selected files:',len(SELECTED))
print('Catalogue size, MiB:',round(sum(r['catalogue_size_bytes'] for r in SELECTED)/1048576,2))
print('Expected reference result: approximately 60 files, with h25v08 and h26v08. Review unexpected differences.')


## 6. Authenticate and verify one science download

This downloads a single January 2001 Terra file to Colab's working disk, checks its HDF4 signature and catalogue size, and reads its dataset structure. The verified file and a nonsecret receipt are then copied to Drive. Passwords are entered only in the login prompt and are not persisted.

This is file verification, not zero-month certification. Actual daily masks and quality states will be analyzed in the next stage.

API reference: https://earthaccess.readthedocs.io/en/stable/api/


In [ ]:
import earthaccess, shutil
from pyhdf.SD import SD, SDC

auth=earthaccess.login(strategy='interactive',persist=False)
assert auth.authenticated, 'Earthdata authentication failed.'
SAMPLE=next(r for r in SELECTED if r['name'].startswith('MOD14A1.A2001001.h25v08.'))
paths=earthaccess.download([SAMPLE['download_url']],provider='LPCLOUD',local_path=str(LOCAL),threads=1)
assert len(paths)==1
sample_path=Path(paths[0])
with sample_path.open('rb') as stream:
    assert stream.read(4)==b'\x0e\x03\x13\x01', 'The response is not an HDF4 science file.'
assert abs(sample_path.stat().st_size-SAMPLE['catalogue_size_bytes'])<=2, 'Incomplete or changed science file.'

hdf=SD(str(sample_path),SDC.READ)
try:
    datasets=hdf.datasets()
    assert 'FireMask' in datasets and 'QA' in datasets
    fire_shape=list(hdf.select('FireMask').info()[2])
    qa_shape=list(hdf.select('QA').info()[2])
    assert fire_shape==qa_shape and len(fire_shape)==3 and fire_shape[0]==8
    assert 'StructMetadata.0' in hdf.attributes()
    dataset_metadata={'datasets':{name:list(info[1]) for name,info in datasets.items()},
                      'fire_attributes':hdf.select('FireMask').attributes(),
                      'qa_attributes':hdf.select('QA').attributes()}
finally:hdf.end()

verified_dir=RUN/'raw'/'verified_l3'
verified_dir.mkdir(parents=True,exist_ok=True)
destination=verified_dir/sample_path.name
if destination.exists():
    assert hashlib.sha256(destination.read_bytes()).digest()==hashlib.sha256(sample_path.read_bytes()).digest()
else:shutil.copy2(sample_path,destination)
receipt={'id':SAMPLE['id'],'name':SAMPLE['name'],'bytes':destination.stat().st_size,
         'sha256':hashlib.sha256(destination.read_bytes()).hexdigest(),
         'selection_manifest_sha256':hashlib.sha256(manifest_bytes).hexdigest(),
         'dataset_metadata':dataset_metadata,'pixel_coverage_verified':False}
(STAGE/'first_file_verification.json').write_text(json.dumps(receipt,indent=2))
print('Verified:',receipt['name'])
print('Bytes:',receipt['bytes'])
print('SHA256:',receipt['sha256'])
print('Dataset dimensions:',dataset_metadata['datasets'])
print('Fire-mask attributes:',dataset_metadata['fire_attributes'])
print('Receipt saved to:',STAGE/'first_file_verification.json')


## Stage 2 checkpoint

Share the selection table and the final file-verification output. Expected shapes include `FireMask` and `QA`, each `(8, 1200, 1200)`. Zero Aqua files in the two early January months remain expected.

Next we will download the remaining selected files with retries/checkpoints, validate all files, compare and deduplicate overlapping daily fire-mask/QA bytes, and calculate observation support and fire counts. Conflicting overlaps must stop processing. Cloudy pixels, unknown values and missing inputs remain distinct. We will also reconcile swath acquisitions, including the December 2022 file listed in LAADS but absent from CMR.

No model fitting, NOAA replacement, zero restoration or frontend export is performed by this stage.
